### 문서 로딩 & 청킹

- pdf를 langchain 로더를 활용해서 문서 형태로 변환
- 정리한 페이지를 CSV로 저장해서 -> 재사용도 가능한 형태로 변경


### 문서를 통을 한꺼번에 넣을 경우 생기는 문제
1. 임베딩 모델 입력한도 제한
    - small -> 8000 정도 토큰이 최대
    - 분할 필요

2. 한 문서를 통째로 넣으면 주제가 섞여 있을 때
    - 벡터와 하더라도 주제에 따라서 분류하기가 적절하지 않다
    - 한 페이지에 여러 주제가 있다면 -> 벡터화 했을 떄 어디로?
   - 예) 한 페이지에 
    - 1. 청년 월세 지원, 2. 노인복지금 등 -> 분리 후 벡터화


In [4]:
from pypdf import PdfReader

reader = PdfReader("../data/16-1_K희망사다리2026_모두의정책.pdf")
total_k_ladder = len(reader.pages)

In [5]:
reader.pages[0]

{'/ArtBox': [0.0, 0.0, 430.866, 632.126],
 '/BleedBox': [0.0, 0.0, 430.866, 632.126],
 '/Contents': {'/Filter': '/FlateDecode'},
 '/CropBox': [0.0, 0.0, 430.866, 632.126],
 '/MediaBox': [0.0, 0.0, 430.866, 632.126],
 '/Parent': {'/Count': 4,
  '/Kids': [IndirectObject(1, 0, 2304217263440),
   IndirectObject(27, 0, 2304217263440),
   IndirectObject(29, 0, 2304217263440),
   IndirectObject(26110, 0, 2304217263440)],
  '/Parent': {'/Count': 34,
   '/Kids': [IndirectObject(26109, 0, 2304217263440),
    IndirectObject(26111, 0, 2304217263440),
    IndirectObject(26117, 0, 2304217263440),
    IndirectObject(26123, 0, 2304217263440),
    IndirectObject(26129, 0, 2304217263440),
    IndirectObject(26135, 0, 2304217263440),
    IndirectObject(26141, 0, 2304217263440)],
   '/Parent': {'/Count': 268,
    '/Kids': [IndirectObject(26108, 0, 2304217263440),
     IndirectObject(26147, 0, 2304217263440),
     IndirectObject(26178, 0, 2304217263440),
     IndirectObject(26240, 0, 2304217263440),
     I

In [ ]:
# for i in enumerate(reader.pages[:5]):
#     print()
text = reader.pages[0].extract_text()


In [8]:
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

load_dotenv() #.env 파일에 저장된 api 키 로드

True

In [11]:
pdf_docs = []
for i ,page in enumerate(reader.pages):
    text = page.extract_text()
    pdf_docs.append(Document(page_content=text,
                             metadata={"source":"k희망사다리2026_모두의정책",
                                       "page":i+1}))
    pdf_docs[:10]

#문서 청킹
- RecursiveCharacterTextSpiltter
- 엔터가 두번이다 -> 문단. 1차적 문단에서 먼저 자르고, 엔터기준(줄바꿈)으로 자름
    . 기준으로 자르게 시킬 수 있다.

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=400,
                                          chunk_overlap=80)
chunks = splitter.split_documents(pdf_docs)
len(chunks)

606

In [16]:
print(pdf_docs[9].page_content)

유아 단계적 
무상교육·보육
008따뜻한 동행 모두가 행복한 사회 - 2026년 신규 민생지원 제도
지원대상 	 • 	어린이집·유치원에	다니는	4~5세	유아
핵심내용 	 •	 학부모가	어린이집·유치원에	매월	납부하는	평균	학부모	부담금
  ※  유아교육비·보육료 월 5만 원 추가지원 별도
이용방법 	 • 	별도	신청절차	없이	학부모	납부금에서	차감되며,	어린이집·유치원을	통해	
지원
문의처	 •교육부	상담센터(☎02-6222-6060)
구분 지원금액 핵심내용
어린이집 월 7만 원 기타 필요경비
공립유치원 월 2만 원 방과후과정비
사립유치원 월 11만 원 유아교육비
02-6222-
6060
교육부 상담센터


In [21]:
# 임베딩 및 저장
DB_PATH = "../data/k_ladder_2026"

#임베딩 모델 설정
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 벡터 스토어에 저장
vector_store = Chroma.from_documents(

    documents=pdf_docs,
    embedding=embeddings,
    collection_name="k_ladder_2026",
    persist_directory=DB_PATH
)
embeddings 

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000002184C18ADE0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x0000021849F1D130>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [22]:
# 검색
rag_docs =vector_store.similarity_search("청년 지원 월세는 어떻게 신청하나요", k=10)
rag_docs

[Document(id='6a815eba-4cc7-4ed9-815e-6d12542200c3', metadata={'page': 14, 'source': 'k희망사다리2026_모두의정책'}, page_content='청년미래적금\n 1600-5500\n금융위원회\n012따뜻한 동행 모두가 행복한 사회 - 2026년 신규 민생지원 제도\n지원대상 \t •\t 일정\t소득\t이하\t만\t19~34세\t청년(병역\t최대\t6년\t인정)\n\t \t - \t\t일반형:\t개인\t소득\t6,000만\t원\t이하\t소득자\t또는\t연\t매출\t3억\t원\t이하\t소상공인\t\n중\t가구\t중위소득\t200%\t이하\n\t \t - \t\t우대형:\t개인소득\t3,600만\t원\t이하\t중소기업\t재직자\t또는\t연\t매출\t1억\t원\t\n이하\t소상공인\t중\t가구\t중위소득\t150%\t이하\n   ※  일반형 요건을 충족하는 중소기업 신규 재직자는 우대형 분류\n핵심내용 \t • \t만기\t3년\n\t •\t납입액(월\t50만\t원\t한도)에\t대한\t정부기여금\t지원(일반형\t6%,\t우대형\t12%)\t\n및\t이자소득\t비과세\n  ※  개인소득 6,000~7,500만 원 이하는 이자소득 비과세만 부여\n이용방법 \t • \t신청\t기간:\t2026년\t6월\t이후(추후\t안내\t예정)\n\t •신청\t방법:\t비대면\t가입\t신청(추후\t안내\t예정)\n문의처\t •금융위원회(☎1600-5500) \t및\t서민금융진흥원\n최대 2,000만 원 \n이상\n3년\t만기'),
 Document(id='032f34ef-1d38-44d5-9d39-5b28bbb63aab', metadata={'source': 'k희망사다리2026_모두의정책', 'page': 14}, page_content='청년미래적금\n 1600-5500\n금융위원회\n012따뜻한 동행 모두가 행복한 사회 - 2026년 신규 민생지원 제도\n지원대상 \t •\t 일정\t소득\t이하\t만\t19~34세

In [20]:
print(rag_docs[3].page_content)

238분야별 서비스 - 문화
청년문화예술패스
 1577-1968
청년문화예술패스
고객지원센터
지원대상 	 •	 올해	19~20세가	되는	2006~2007년생	청년
		 ※		생애	최초	1회	지원(2025년	포인트	사용자	신청	불가)
핵심내용 	 •	 1인당	연	15~20만	원	국내	공연·전시·영화	관람비	지원
이용방법 	 •	 2026년	2월	25일(수)부터	6월	30일(화)까지	청년문화예술패스	누리집	회원
가입	후	온라인	신청(지역별	예산	소진	시	마감)
	 •포인트	사용기간은	발급일로부터	2026년	12월	31일(목)까지(관람일	기준)
		 ※		신청	후	예매일	기준	2026년	7월	31일(금)까지	사용금액이	없는	이용자	지원금	회수
문의처	 •청년문화예술패스	고객지원센터(☎1577-1968)
청년문화예술패스 누리집 회원가입 및 로그인
본인인증 및 신청
신청자격 검증
청년문화예술패스 발급
협력예매처에서 온라인 예매 시 지원금 사용
